# 02: stationary sensor calibration

Day 1 checked axes and onboard state estimates. E2 (45 minutes) examines IMU bias and noise. Use live data, checked recordings or simulation; choose a gyro correction and test it on held-out data. Run setup in a fresh kernel. Results: `data/<run_id>/`.


In [ ]:
from pathlib import Path
import json
import hashlib
import os
import sys

# Shared helpers are beside day2; data stays beside these notebooks.
notebook_dir = Path.cwd()
if (notebook_dir / "code/day2").is_dir():
    notebook_dir = notebook_dir / "code/day2"
sys.path.insert(0, str(notebook_dir.resolve().parent))

import numpy as np
from IPython.display import display
from day2.notebook_tools import (
    capture, completed_capture, read_imu, plot_imu_calibration,
    simulated_imu_pair, display_table, next_attempt, drone_uri,
)

run_id = "trial-01"  # Label this session; each comparison gets a fresh attempt folder.
output = notebook_dir / "data" / run_id
output.mkdir(parents=True, exist_ok=True)
print("Results:", output.resolve())

LABEL = None  # Set the sticker label for live data: 2 or "A", as on Day 1.
uri = drone_uri(LABEL) if LABEL is not None else os.environ.get("CF_URI", "")


Set `LABEL` to the sticker label, as on Day 1. With `LABEL=None`, `CF_URI` can supply the assigned URI. Close cfclient before capture. One laptop records; share files within the team.


## E2: check calibration before tuning — 45 minutes

Record the accelerometer and gyroscope channels of the **inertial measurement unit (IMU)** while the drone rests on a stable surface. Use two separate 60-second files: one to estimate gyro offsets and variability, and one to check that correction. Accelerometer readings are body-frame specific force in g; gyro readings are degrees/s and may already include onboard compensation.

Choose **one** recording route: cfclient (E2A) or the notebook (E2B). One laptop connects to the drone; share the saved files within the team. Reuse Day 1’s drone identity, axes, units and drift observations; check changes and identify the raw IMU channels (5 minutes). Allow 10 minutes to record, 15 to calculate, 10 for the held-out check and 5 to save the result.

### E2A: acquire the IMU files with cfclient

1. Place the drone on a stable, undisturbed surface. Connect to the assigned URI. In **Parameters > stabilizer > stop**, enter `1`, press Enter and check the displayed value. Leave the stop engaged and keep clear of propellers.
2. In **Settings > Logging configuration > Default > Create config**, create an IMU block with six float channels: `acc.x`, `acc.y`, `acc.z`, `gyro.x`, `gyro.y`, `gyro.z`. Set **Logging period** to **20 ms** and save; check names/types in **Log TOC**. Create a Motors block with `motor.m1`, `motor.m2`, `motor.m3`, `motor.m4` at 50 ms if needed.
3. In **Log Blocks**, start Motors and check all four traces are zero in **Plotter**. Enable **Start** and **Write to file** for IMU. Record 60 seconds at rest, stop writing, then make a **separate 60-second validation recording** in the same pose.
4. Use **Settings > Open config folder** to find `logdata/` and its timestamped recordings. Retain both CSVs and note firmware, decks and conditions. Disconnect cfclient before any Python capture.

### E2B: record the IMU files with Python

Place the drone on a stable surface, keep clear of propellers and close other radio clients. Set `ready_for_motor_stop = True` and `run_imu_capture = True` below. The helper sets `stabilizer.stop=1` and verifies the firmware lock and zero motor outputs before recording. Leave the drone at rest.

Run once with `capture_filename = "calibration.csv"`, then again with `"validation.csv"`. Each file records 60 seconds after the first IMU packet, at a requested 20 ms period, in `data/<run_id>/E2B/`. Existing files are preserved; choose a new name or run ID to repeat. Partial captures remain available for diagnosis but are excluded from automatic selection.

**After either route:** the stop remains engaged until power-cycle. Place the drone down, power-cycle it and repeat preflight checks before flight. If acquisition blocks progress, follow the five-minute fallback rule above; mark calibration unfinished when using the supplied filter assumptions.


In [ ]:
run_imu_capture = False  # Opt in for a stationary recording.
ready_for_motor_stop = False  # Drone on surface; propellers clear; other radio clients closed.
capture_filename = "calibration.csv"  # Then validation.csv for a separate recording.
channels = (
    "acc.x", "acc.y", "acc.z",
    "gyro.x", "gyro.y", "gyro.z",
)

if run_imu_capture:
    if not ready_for_motor_stop:
        raise ValueError("Confirm the drone is on a stable surface and other radio clients are closed.")
    saved_imu, sample_count = capture(
        uri, output / "E2B" / capture_filename,
        channels=channels, duration_s=60, period_ms=20)
    print(f"Saved {sample_count} IMU samples to {saved_imu}")
else:
    print("IMU capture skipped. Select checked stationary files below, or continue offline.")


### E2C: calculate and validate

Select `calibration_csv` and separate `validation_csv`; unchanged, completed E2B files are selected automatically. Enter cfclient paths explicitly. Set `stationary_checks_passed = True` after verifying firmware stop, zero motor outputs and undisturbed rest.

If acquisition stalls, spend at most five minutes troubleshooting, then set `use_simulated_imu = True`. The helper creates independent, labelled calibration/validation files under `E2-simulated/`. Follow the same analysis; mark actual-drone calibration outstanding.

At rest the gyro reference is zero: estimate its offset from the mean, then freeze it for validation. Predict the mean and RMS of $(y-\bar y_{cal})/s_{cal}$, where $s_{cal}^2=N^{-1}\sum(y-\bar y_{cal})^2$. Calibration values are **zero and one by construction**, regardless of distribution shape. Held-out samples use the **same frozen calibration mean and RMS**: their histogram tests transfer, while calibration shows shape/tails. Interpret one channel against the dashed N(0,1) reference. Shared systematic errors can remain hidden.

Accelerometer components contain gravity projected onto body axes. Check magnitude near **1 g** and keep the same pose for validation; separating bias from tilt requires known additional orientations.

**Completion:** explain the held-out gyro correction or retained compensation, and the gravity check. Broader sensor characterization is optional.


In [ ]:
calibration_csv = None  # Or select a checked stationary IMU CSV.
validation_csv = None   # A separate stationary recording.
stationary_checks_passed = False  # After the motor-stop and undisturbed-rest checks.
use_simulated_imu = False  # Explicit E2 fallback; creates a labelled pair below.

# Invalidate earlier results before trying another pair of files.
channel_values = validation_values = None
gyro_bias = gravity_norm_g = corrected = calibration_summary = None
standardized_calibration = standardized_validation = None
calibration_source = "selected recordings"
if use_simulated_imu:
    calibration_csv, validation_csv = simulated_imu_pair(output / "E2-simulated")
    calibration_source = "simulation; calibration of the actual drone remains outstanding"
print("Calibration source:", calibration_source)
if calibration_csv is None:
    calibration_csv = completed_capture(output / "E2B" / "calibration.csv")
if validation_csv is None:
    validation_csv = completed_capture(output / "E2B" / "validation.csv")

if (stationary_checks_passed or use_simulated_imu) and calibration_csv is not None:
    if validation_csv is not None and Path(calibration_csv).samefile(validation_csv):
        raise ValueError("Calibration and held-out validation need two separate recording files.")
    training_values = read_imu(calibration_csv)
    if validation_csv is not None:
        validation_values = read_imu(validation_csv)
    channel_values = training_values  # Publish the selection only after both files load.
else:
    print("Calibration skipped. Select a checked IMU file and confirm the stationary checks.")


In [ ]:
gyro_bias = gravity_norm_g = None
standardized_calibration = standardized_validation = None
if channel_values is not None:
    channel_names = tuple(f"{sensor}.{axis}" for sensor in ("acc", "gyro") for axis in "xyz")
    acceleration = channel_values[:, :3]  # Body specific force [g].
    training = np.deg2rad(channel_values[:, 3:])
    gyro_bias = training.mean(axis=0)
    print("Calibration mean [rad/s]:", gyro_bias)
    print("Calibration covariance [(rad/s)^2]:", np.cov(training, rowvar=False, ddof=1))
    # Compare samples directly: mean-rounding can give constant decimals a nonzero RMS.
    varying = np.any(channel_values != channel_values[0], axis=0)
    channel_means = channel_values.mean(axis=0)
    channel_means[~varying] = channel_values[0, ~varying]
    centered = channel_values - channel_means
    sigma = np.sqrt(np.mean(centered**2, axis=0))
    standardized_calibration = np.full_like(centered, np.nan)
    standardized_calibration[:, varying] = centered[:, varying] / sigma[varying]
    if validation_values is not None:
        # Freeze both quantities from calibration; never recenter or rescale held-out data.
        standardized_validation = np.full_like(validation_values, np.nan)
        standardized_validation[:, varying] = (validation_values[:, varying] - channel_means[varying]) / sigma[varying]
        print("Held-out standardized mean/std (varying channels):",
              standardized_validation[:, varying].mean(axis=0),
              standardized_validation[:, varying].std(axis=0))
    if not np.all(varying):
        print("Channels with zero measured variation:",
              [name for name, valid in zip(channel_names, varying) if not valid])
    gravity_norm_g = np.linalg.norm(acceleration, axis=1)
    print("Stationary specific-force magnitude: mean/std [g]:",
          gravity_norm_g.mean(), gravity_norm_g.std(ddof=1))


In [ ]:
if gyro_bias is not None:
    plot_imu_calibration(channel_values, channel_names, standardized_calibration, standardized_validation)


In [ ]:
corrected = None
if gyro_bias is not None and validation_values is not None:
    import matplotlib.pyplot as plt

    validation = np.deg2rad(validation_values[:, 3:])
    corrected = validation - gyro_bias  # Keep the training bias fixed.
    print("Held-out mean before correction [rad/s]:", validation.mean(axis=0))
    print("Held-out mean after correction [rad/s]:", corrected.mean(axis=0))
    print("Held-out std before/after [rad/s]:",
          validation.std(axis=0, ddof=1), corrected.std(axis=0, ddof=1))
    figure, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
    for index, axis in enumerate(axes):
        axis.plot(validation[:, index], label="held-out")
        axis.plot(corrected[:, index], label="training bias subtracted")
        axis.set_ylabel(f"gyro {'xyz'[index]} [rad/s]")
    axes[0].legend()
    axes[-1].set_xlabel("sample index in validation recording")
    figure.tight_layout()
    plt.show()
elif gyro_bias is not None:
    print("Select a separate validation recording to complete the held-out check.")


### Save the calibration decision

Save the decision and held-out evidence below. E3 applies this calibration method to acceleration and range; the gyro result remains a diagnostic. The drone retains its calibration.


In [ ]:
calibration_decision = "unfinished"  # Replace with your held-out comparison and decision.
calibration_summary = None
if gyro_bias is not None:
    calibration_summary = {
        "decision": calibration_decision,
        "source": calibration_source,
        "gyro_bias_rad_s": gyro_bias.tolist(),
        "gyro_covariance_rad2_s2": np.cov(training, rowvar=False, ddof=1).tolist(),
        "gravity_mean_g": float(gravity_norm_g.mean()),
        "validation_completed": corrected is not None,
        "held_out_corrected_mean_rad_s": None if corrected is None else corrected.mean(axis=0).tolist(),
        "source_sha256": {
            str(Path(path).resolve()): hashlib.sha256(Path(path).read_bytes()).hexdigest()
            for path in (calibration_csv, validation_csv) if path is not None
        },
    }
    calibration_output = next_attempt(output / "E2")
    calibration_output.mkdir()
    (calibration_output / "calibration.json").write_text(
        json.dumps(calibration_summary, indent=2) + "\n", encoding="utf-8")
    print("Saved calibration decision:", calibration_output / "calibration.json")


Continue with [03 height and velocity](03_height_velocity.ipynb). E3 estimates acceleration offset and range precision from initial rest; E6 tests those frozen values on fresh data.
